## Desafío 3

##### Configuración inicial

In [76]:
import pandas as pd
from tensorflow import keras
from keras import Input
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.utils import pad_sequences
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr


### Datos
Utilizaré como dataset letras de temas de Charly Garcia

In [77]:
df = pd.read_csv('datasets/letras_charly.csv', usecols=['verso']).dropna().drop_duplicates()

print("Cantidad de versos:", df.shape[0])
df.head()

Cantidad de versos: 5454


,verso
0,Podés pasear en limousine
1,cortar las flores del jardín
2,podés cambiar el sol
3,y esconderte si no quieres verme.
4,Puedes ver amanecer


In [78]:
vocabulario = {' ': 1, 'e': 2, 'a': 3, 'o': 4, 's': 5, 'n': 6, 'r': 7, 'i': 8, 'l': 9, 't': 10, 'u': 11, 'd': 12, 'm': 13,
               'c': 14, 'p': 15, 'y': 16, 'v': 17, 'h': 18, 'q': 19, 'g': 20, 'b': 21, ',': 22, '.': 23, 'f': 24, 'á': 25,
               'í': 26, 'ó': 27, 'z': 28, 'é': 29, 'j': 30, 'w': 31, 'k': 32, 'ñ': 35, '?': 36, 'x': 37, 'ú': 38, '!': 39,
               '"': 40, '¿': 41, "'": 42, ')': 43, '(': 44, '’': 45, ':': 46, '-': 49, '¡': 50, '/': 59, 'ü': 60}


def limpiar_texto(texto):
    caracteres_validos = set(vocabulario.keys())
    texto_limpio = ''.join([c for c in texto.lower() if c in caracteres_validos])
    return texto_limpio

In [87]:
corpus = []

for _, row in df.iterrows():
    verso_limpio = limpiar_texto(row.iloc[0])  # Limpiar caracteres no válidos
    palabras = text_to_word_sequence(verso_limpio)  # Tokenizar en palabras válidas

    if not palabras:
        continue  # saltar si quedó vacío

    verso = ' '.join(palabras) + ' '  # reconstruir verso con espacios y uno final
    verso_lista = list(verso)         # convertir el string en lista de caracteres

    corpus.append(verso_lista)

print(f"El corpus posee {len(corpus)} documentos (versos). \nPor ejemplo: {corpus[0:2]}")


El corpus posee 5453 documentos (versos). 
Por ejemplo: [['p', 'o', 'd', 'é', 's', ' ', 'p', 'a', 's', 'e', 'a', 'r', ' ', 'e', 'n', ' ', 'l', 'i', 'm', 'o', 'u', 's', 'i', 'n', 'e', ' '], ['c', 'o', 'r', 't', 'a', 'r', ' ', 'l', 'a', 's', ' ', 'f', 'l', 'o', 'r', 'e', 's', ' ', 'd', 'e', 'l', ' ', 'j', 'a', 'r', 'd', 'í', 'n', ' ']]


In [88]:
length_secuence = [len(secuence) for secuence in corpus]
plt.hist(length_secuence,bins=10)

(array([8.340e+02, 3.087e+03, 1.369e+03, 1.400e+02, 1.200e+01, 8.000e+00,
        1.000e+00, 0.000e+00, 1.000e+00, 1.000e+00]),
 array([  3. ,  16.7,  30.4,  44.1,  57.8,  71.5,  85.2,  98.9, 112.6,
        126.3, 140. ]),
 <BarContainer object of 10 artists>)

Ahora tokenizaré cada una de las letras

In [162]:
max_context_size = 50

In [163]:
tok = Tokenizer()
tok.fit_on_texts(corpus)
tokenized_corpus = tok.texts_to_sequences(corpus)

print(tokenized_corpus[0:9])

[[15, 4, 12, 27, 5, 1, 15, 3, 5, 2, 3, 7, 1, 2, 6, 1, 9, 8, 13, 4, 11, 5, 8, 6, 2, 1], [14, 4, 7, 10, 3, 7, 1, 9, 3, 5, 1, 22, 9, 4, 7, 2, 5, 1, 12, 2, 9, 1, 28, 3, 7, 12, 24, 6, 1], [15, 4, 12, 27, 5, 1, 14, 3, 13, 21, 8, 3, 7, 1, 2, 9, 1, 5, 4, 9, 1], [16, 1, 2, 5, 14, 4, 6, 12, 2, 7, 10, 2, 1, 5, 8, 1, 6, 4, 1, 19, 11, 8, 2, 7, 2, 5, 1, 17, 2, 7, 13, 2, 1], [15, 11, 2, 12, 2, 5, 1, 17, 2, 7, 1, 3, 13, 3, 6, 2, 14, 2, 7, 1], [14, 4, 6, 1, 14, 3, 17, 8, 3, 7, 1, 12, 2, 5, 12, 2, 1, 11, 6, 1, 18, 4, 10, 2, 9, 1], [16, 1, 6, 4, 1, 10, 8, 2, 6, 2, 5, 1, 11, 6, 1, 15, 4, 19, 11, 8, 10, 4, 1, 12, 2, 1, 3, 13, 4, 7, 1, 15, 3, 7, 3, 1, 12, 3, 7, 1], [16, 2, 6, 12, 4, 1, 12, 2, 1, 9, 3, 1, 14, 3, 13, 3, 1, 3, 9, 1, 9, 8, 17, 8, 6, 20, 1], [5, 8, 2, 6, 10, 2, 5, 1, 2, 9, 1, 2, 6, 14, 8, 2, 7, 7, 4, 1]]


Separamos sets de entrenamiento y validación

In [164]:
train, val, _, _ = train_test_split(tokenized_corpus, tokenized_corpus, test_size=0.2, random_state=42)
train


[[16,
  1,
  19,
  11,
  2,
  1,
  6,
  4,
  1,
  15,
  8,
  2,
  6,
  5,
  3,
  5,
  1,
  7,
  2,
  20,
  7,
  2,
  5,
  3,
  7,
  1,
  15,
  4,
  7,
  19,
  11,
  2,
  1,
  10,
  2,
  1,
  18,
  3,
  5,
  1,
  8,
  12,
  4,
  1,
  13,
  8,
  1,
  3,
  13,
  4,
  7,
  1],
 [3, 1, 5, 2, 13, 21, 7, 3, 7, 1, 2, 5, 2, 1, 14, 3, 13, 8, 6, 4, 1],
 [12, 2, 1, 13, 8, 1, 7, 4, 14, 30, 1, 3, 6, 12, 1, 7, 4, 9, 9, 1, 16, 4, 1],
 [18,
  4,
  15,
  2,
  1,
  16,
  4,
  11,
  1,
  20,
  11,
  2,
  5,
  5,
  1,
  13,
  16,
  1,
  6,
  3,
  13,
  2,
  1],
 [6,
  3,
  14,
  2,
  1,
  11,
  6,
  3,
  1,
  22,
  9,
  4,
  7,
  1,
  10,
  4,
  12,
  4,
  5,
  1,
  9,
  4,
  5,
  1,
  12,
  24,
  3,
  5,
  1,
  5,
  3,
  9,
  2,
  1,
  2,
  9,
  1,
  5,
  4,
  9,
  1],
 [6,
  2,
  6,
  3,
  1,
  16,
  4,
  1,
  17,
  4,
  16,
  1,
  3,
  13,
  3,
  7,
  10,
  2,
  1,
  12,
  2,
  1,
  17,
  2,
  7,
  12,
  3,
  12,
  1],
 [16, 4, 1, 2, 7, 3, 1, 11, 6, 1, 18, 4, 13, 21, 7, 2, 1, 21, 11, 2, 6, 4, 1],
 [10, 

Realizamos un padding para ajustar al máximo valor de contexto

In [165]:
def seq_of(set, data_aug=True):
    train_seq = []

    for word in set:
        if data_aug:
            subseq =  [word[:i+2] for i in range(len(word)-1)]
        else:
            subseq = [word]
        train_seq.append(pad_sequences(subseq, maxlen=max_context_size+1, padding='pre'))
    train_seqs = np.concatenate(train_seq, axis=0)

    return train_seqs, train_seqs[:, :-1], train_seqs[:, -1]

In [166]:
train_seqs, X, y = seq_of(train)
print(f'training shape {train_seqs.shape}')
for i in range(100):
    print(f'(X): {X[i]} ({ "".join([ tok.index_word.get(c,"") for c in X[i]]) }) (y): {y[i]} ({tok.index_word[y[i]]})')

training shape (107948, 51)
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0 16] (y) (y): 1 ( )
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 16  1] (y ) (y): 19 (q)
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 16
  1 19] (y q) (y): 11 (u)
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 16  1
 19 11] (y qu) (y): 2 (e)
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 16  1 19
 11  2] (y que) (y): 1 ( )
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0

In [167]:
vocab_size = len(tok.word_counts)
print(f'Tamaño del vocabulario: {vocab_size} \n')
print(f'Vocabulario: {tok.word_index} \n')
print(f'Vocabulario por apariciones: {tok.word_docs}')

Tamaño del vocabulario: 38 

Vocabulario: {' ': 1, 'e': 2, 'a': 3, 'o': 4, 's': 5, 'n': 6, 'r': 7, 'i': 8, 'l': 9, 't': 10, 'u': 11, 'd': 12, 'm': 13, 'c': 14, 'p': 15, 'y': 16, 'v': 17, 'h': 18, 'q': 19, 'g': 20, 'b': 21, 'f': 22, 'á': 23, 'í': 24, 'ó': 25, 'z': 26, 'é': 27, 'j': 28, 'w': 29, 'k': 30, 'ñ': 31, 'x': 32, 'ú': 33, '¿': 34, "'": 35, '’': 36, '¡': 37, 'ü': 38} 

Vocabulario por apariciones: defaultdict(<class 'int'>, {'d': 3129, 'é': 445, 'e': 5034, 'i': 3629, 'o': 4567, 'r': 4004, 'm': 2766, 'p': 2004, ' ': 5453, 'a': 4702, 'l': 3250, 'u': 3448, 'n': 4213, 's': 4121, 'f': 729, 'c': 2463, 'j': 466, 'í': 590, 't': 3412, 'b': 1166, 'y': 1891, 'q': 1410, 'v': 1426, 'h': 1320, 'g': 1250, 'ó': 480, 'ñ': 146, 'z': 460, 'á': 612, 'ú': 71, 'x': 81, 'w': 284, 'k': 196, '¿': 41, '¡': 12, 'ü': 2, '’': 30, "'": 32})


### Modelo

In [168]:
model = Sequential()

model.add(Embedding(input_dim=vocab_size+1, output_dim=50, input_shape=(max_context_size,)))

# Bidirectional como primera LSTM (con secuencia)
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.2))

# Segunda LSTM (unidireccional, resume)
model.add(LSTM(64))

model.add(Dense(32, activation='relu'))
model.add(Dense(vocab_size+1, activation='softmax'))

model.compile(loss=SparseCategoricalCrossentropy(), optimizer='adam', metrics=['accuracy'])

model.summary()


/Users/christianpisani/Learning/Posgrado/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 50, 50)         │         1,950 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 50, 128)        │        58,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 39)             │         1,287 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 113,605 (443.77 KB)

 Trainable params: 113,605 (443.77 KB)

 Non-trainable params: 0 (0.00 B)

In [169]:
from tensorflow.keras.callbacks import Callback

class PplCallback(Callback):
    def __init__(self, val_data, max_context_size):
        super().__init__()
        self.max_context_size = max_context_size
        self.X_val, self.y_val = self._preparar_datos(val_data)

    def _preparar_datos(self, corpus):
        X = []
        y = []

        for seq in corpus:
            for i in range(1, len(seq)):
                contexto = seq[max(0, i - self.max_context_size):i]
                target = seq[i]

                contexto_ids = [c for c in contexto]
                contexto_ids = [0] * (self.max_context_size - len(contexto_ids)) + contexto_ids

                X.append(contexto_ids)
                y.append(target)

        return np.array(X), np.array(y)

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}

        val_loss, val_acc = self.model.evaluate(self.X_val, self.y_val, verbose=0)
        val_ppl = np.exp(val_loss)

        logs['val_loss'] = val_loss
        logs['val_accuracy'] = val_acc
        logs['val_perplexity'] = val_ppl

        print(f'\n📏 Validation loss: {val_loss:.4f} | accuracy: {val_acc:.4f} | perplexity: {val_ppl:.4f}')


In [173]:
from keras.src.callbacks import EarlyStopping

_, X_val, y_val =seq_of(val)

ppl_callback = PplCallback(val_data=val, max_context_size=max_context_size)
early_stopping = EarlyStopping(monitor='val_perplexity', patience=5, restore_best_weights=True, mode='min')

hist = model.fit(X, y, epochs=20, callbacks=[ppl_callback, early_stopping], batch_size=256)

Epoch 1/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.3929 - loss: 1.9733
📏 Validation loss: 1.9490 | accuracy: 0.4018 | perplexity: 7.0215
422/422 ━━━━━━━━━━━━━━━━━━━━ 62s 148ms/step - accuracy: 0.3930 - loss: 1.9732 - val_loss: 1.9490 - val_accuracy: 0.4018 - val_perplexity: 7.0215
Epoch 2/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - accuracy: 0.4084 - loss: 1.9180
📏 Validation loss: 1.8972 | accuracy: 0.4142 | perplexity: 6.6671
422/422 ━━━━━━━━━━━━━━━━━━━━ 62s 147ms/step - accuracy: 0.4084 - loss: 1.9179 - val_loss: 1.8972 - val_accuracy: 0.4142 - val_perplexity: 6.6671
Epoch 3/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.4167 - loss: 1.8745
📏 Validation loss: 1.8552 | accuracy: 0.4265 | perplexity: 6.3928
422/422 ━━━━━━━━━━━━━━━━━━━━ 62s 146ms/step - accuracy: 0.4167 - loss: 1.8745 - val_loss: 1.8552 - val_accuracy: 0.4265 - val_perplexity: 6.3928
Epoch 4/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - accuracy: 0.4319 - loss: 1.8324
📏 Validation lo

In [109]:
# Entrenamiento
epoch_count = range(1, len(hist.history['perplexity']) + 1)
sns.lineplot(x=epoch_count,  y=hist.history['perplexity'], label='train')
#sns.lineplot(x=epoch_count,  y=hist.history['val_loss'], label='valid')
plt.show()

KeyError: 'perplexity'

In [129]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def generate_seq(model, tokenizer, seed_text, max_length, n_chars=100, temperature=1.0):
    result = list(seed_text.lower())

    for _ in range(n_chars):
        # Convertir contexto a índices
        encoded = [tokenizer.word_index.get(c, 0) for c in result[-max_length:]]
        encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

        # Obtener predicciones
        preds = model.predict(encoded, verbose=0)[0]

        # Aplicar temperatura
        preds = np.log(preds + 1e-10) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)

        # Elegir siguiente carácter con muestreo probabilístico
        next_index = np.random.choice(len(preds), p=preds)
        next_char = tokenizer.index_word.get(next_index, '')

        result.append(next_char)

    return ''.join(result)


In [174]:
input_text='los dinosaurios'

print(generate_seq(model, tok, input_text, max_length=max_context_size, n_chars=100, temperature=0.2))

los dinosaurios a la cama de la vez de la mis de puedes de la mi amor de la mestrar de la comprando la mestrar de l


Si bien el modelo logra predecir el siguiente caracter no logré que las secuencias creadas construyan palabras que tengan sentido